# Does authority spreading before founders leave keep projects alive?

This notebook reimplements the **Degree-of-Authorship / Truck-Factor / Truck-Factor-Detachment-Departure (TFDD)** pipeline from Avelino et al. (ESEM 2019) and adds a *new* covariate: how spread out commit authority was in the 6-12 months **before** a founder's departure.

For each repository the pipeline:
1. Resolves author aliases (email/github-login normalization).
2. Computes yearly cumulative Degree-of-Authorship (DOA) per file per author, using Fritz et al.'s weights.
3. Derives the yearly greedy Truck-Factor (TF) set.
4. Detects TFDD events (TF-set fully silent for 12 months) and isolates *founder-only* TFDDs.
5. Computes the pre-departure authority-diffusion trajectory (founder commit-share + distinct non-founder DOA owners) alongside Avelino et al.'s original at-TFDD snapshot covariates.
6. Classifies 18-month post-TFDD survival (thriving/maintained/dormant/dead).
7. Runs a matched-pairs bootstrap comparison, BH-corrected logistic + ordinal regressions, and a window-shuffle placebo check.

Because the real GitHub-mined corpus this pipeline was run on is tiny (only 6 founder-only TFDD events survived upstream API rate limiting), this demo instead runs the **exact same pipeline code** against the project's own synthetic self-test repositories (`make_synthetic_repos` from `method.py`) — this is the same smoke-test data the original run used to validate the mechanics, and it produces enough events to show every stage of the pipeline working end-to-end, including the regression and placebo checks that the real run could not reach.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# psutil, loguru — NOT pre-installed on Colab, always install
_pip('psutil==7.1.4')
_pip('loguru==0.7.3')

# numpy, pandas, scipy, scikit-learn, statsmodels — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0')

In [ ]:
from __future__ import annotations

import gc
import json
import random
import sys
import time
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests

# matplotlib added for the results/visualization cell at the end of this notebook
import matplotlib.pyplot as plt

try:
    from statsmodels.miscmodels.ordinal_model import OrderedModel
except Exception:  # pragma: no cover
    OrderedModel = None

# Fritz et al. DOA weights, as used by Avelino et al. (ESEM 2019)
DOA_FA, DOA_LOG, DOA_AC = 3.293, 1.098, -1.017
MONTH = timedelta(days=30.4375)


def months(n: float) -> timedelta:
    return n * MONTH

## Loading the demo data

`mini_demo_data.json` is a curated set of 8 **synthetic** repositories generated by `method.py`'s own `make_synthetic_repos()` self-test helper (used in the original run for smoke-testing). Half of the repos get a co-maintainer handoff before the founder goes silent (diffuse authority), the other half do not — so the demo has both surviving and non-surviving founder-departure events to analyze.

The loader below tries the GitHub-hosted copy first (for Colab), then falls back to the local file (for running this notebook next to its data file).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-1/experiment-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
raw_repos = load_data()
print(f"Loaded {len(raw_repos)} raw synthetic repo records")
print(json.dumps(raw_repos[0]["repo_metadata"], indent=2))
print(f"repo 0 has {len(raw_repos[0]['commits'])} commit records")

## Config

All tunable parameters from the original `method.py`, gathered here. Values start at the minimum needed to get founder-TFDD events out of the small demo corpus, and can be scaled up (e.g. `N_BOOTSTRAP`, `N_PLACEBO_DRAWS`) toward the original run's values as time allows.

In [ ]:
# --- config: start at the minimum values needed to see the pipeline work end-to-end ---
MAX_REPOS = len(raw_repos)     # how many repos from the demo data to process (all 8)
SILENCE_MONTHS = 12            # TF-set must be silent this many months to count as a TFDD
SURVIVAL_WINDOW_MONTHS = 18    # post-TFDD window used to classify survival
PRE_WINDOW_FAR_MONTHS = 12     # start of the pre-departure diffusion window
PRE_WINDOW_NEAR_MONTHS = 6     # end of the pre-departure diffusion window
N_PLACEBO_DRAWS = 5            # window-shuffle placebo draws per repo (original: 500)
N_BOOTSTRAP = 200              # bootstrap resamples for matched-pairs CI (original: 10_000)
RNG_SEED = 20260820

## Step 0: data loading + alias resolution

Adapt a raw repo record into a normalized commits `DataFrame` (one row per file touched per commit), collapsing author aliases onto a canonical `author_id` via normalized email / github-login matching, and dropping bulk-import first commits (Kalliamvakou et al. 2014 heuristic: a commit within the first 7 days touching >80% of the eventual file set).

In [ ]:
def _normalize_email(email: str) -> str:
    email = (email or "").strip().lower()
    if "@" in email:
        local, domain = email.rsplit("@", 1)
        local = local.split("+", 1)[0]
        if domain == "users.noreply.github.com":
            # e.g. 12345+login@users.noreply.github.com -> login
            if "+" in local:
                local = local.split("+", 1)[1]
            return f"github:{local}"
        return f"{local}@{domain}"
    return email


def resolve_aliases(commits: pd.DataFrame) -> tuple[pd.Series, float]:
    """Collapse (name, email) pairs onto a canonical author_id.

    Primary key: normalized email (or github login where resolvable via the
    noreply-email convention). Falls back to normalized display name when no
    email is present. Returns (author_id series, collapse_rate)."""
    email_norm = commits.get("author_email", pd.Series([""] * len(commits))).fillna("").map(_normalize_email)
    name_norm = commits.get("author_name", pd.Series([""] * len(commits))).fillna("").str.strip().str.lower()
    login = commits.get("author_login", pd.Series([None] * len(commits)))

    author_id = login.where(login.notna() & (login.astype(str).str.len() > 0), None)
    author_id = author_id.fillna(email_norm.where(email_norm.str.len() > 0, None))
    author_id = author_id.fillna(name_norm)
    author_id = author_id.replace("", "unknown")

    n_raw = commits.get("author_email", email_norm).fillna(commits.get("author_name", name_norm)).nunique()
    n_resolved = author_id.nunique()
    collapse_rate = 0.0 if n_raw == 0 else max(0.0, (n_raw - n_resolved) / n_raw)
    return author_id.astype(str), collapse_rate


def _detect_import_artifact_files(commits: pd.DataFrame) -> pd.DataFrame:
    """Flag and drop bulk-import first commits (Kalliamvakou et al. 2014):
    a commit within the first 7 days touching >80% of the eventual repo's
    file set is almost certainly a migrated-history import, not real
    founder authorship."""
    if commits.empty:
        return commits
    t0 = commits["ts"].min()
    early = commits[commits["ts"] <= t0 + timedelta(days=7)]
    total_files = commits["file"].nunique()
    if total_files == 0:
        return commits
    bad_shas = set()
    for sha, grp in early.groupby("sha"):
        if grp["file"].nunique() / total_files > 0.80 and len(early["sha"].unique()) > 1:
            bad_shas.add(sha)
    if bad_shas:
        commits = commits[~commits["sha"].isin(bad_shas)]
    return commits


def load_repo_commits(raw_repo: dict) -> Optional[dict]:
    """Adapt one dataset-dependency repo record into a normalized dict with
    a commits DataFrame (sha, author_id, ts, file) and repo metadata."""
    meta = raw_repo.get("repo_metadata", raw_repo.get("metadata", raw_repo))
    commit_records = raw_repo.get("commits", raw_repo.get("commit_log", []))
    if not commit_records:
        return None

    rows = []
    for c in commit_records:
        ts_raw = c.get("timestamp") or c.get("committer_date") or c.get("date") or c.get("ts")
        try:
            ts = pd.to_datetime(ts_raw, utc=True)
        except Exception:
            continue
        sha = c.get("sha") or c.get("commit_sha") or c.get("hash")
        author_email = c.get("author_email") or c.get("email")
        author_name = c.get("author_name") or c.get("name")
        author_login = c.get("author_login") or c.get("login")
        files = c.get("files_touched") or c.get("files") or c.get("files_changed") or []
        if isinstance(files, dict):
            files = list(files.keys())
        if not files:
            continue
        for f in files:
            fpath = f.get("path") if isinstance(f, dict) else f
            if not fpath:
                continue
            rows.append(
                dict(
                    sha=sha,
                    ts=ts,
                    author_email=author_email,
                    author_name=author_name,
                    author_login=author_login,
                    file=fpath,
                )
            )
    if not rows:
        return None
    commits = pd.DataFrame(rows)
    commits["author_id"], collapse_rate = resolve_aliases(commits)
    commits = commits.sort_values("ts").reset_index(drop=True)
    commits = _detect_import_artifact_files(commits)
    if commits.empty:
        return None

    repo_id = meta.get("full_name") or meta.get("name") or raw_repo.get("repo") or raw_repo.get("id") or "unknown/unknown"
    stars = float(meta.get("stars", meta.get("stargazers_count", 0)) or 0)
    forks = float(meta.get("forks", meta.get("forks_count", 0)) or 0)
    language = meta.get("language") or "unknown"
    license_ = meta.get("license") or "unknown"
    if isinstance(license_, dict):
        license_ = license_.get("key", license_.get("name", "unknown"))

    return dict(
        repo_id=str(repo_id),
        commits=commits,
        stars=stars,
        forks=forks,
        language=str(language),
        license=str(license_),
        alias_collapse_rate=collapse_rate,
    )

## Step 1-2: yearly DOA table + Truck Factor set

`compute_doa_owner_per_file` picks the primary DOA (Degree-of-Authorship) owner per file using all commits up to a given date (Fritz et al. weights: first-author bonus + log(own commits) - log(others' commits)). `truck_factor_set` then greedily picks the smallest set of authors that together own >=50% of files — this is the repo's Truck Factor.

In [ ]:
def compute_doa_owner_per_file(commits: pd.DataFrame, as_of: pd.Timestamp) -> dict[str, str]:
    """Primary DOA owner per file, using all commits up to `as_of` (cumulative
    window, matching Avelino et al.'s yearly-snapshot design)."""
    window = commits[commits["ts"] <= as_of]
    if window.empty:
        return {}
    owners: dict[str, str] = {}
    for fpath, grp in window.groupby("file"):
        grp_sorted = grp.sort_values("ts")
        first_author = grp_sorted.iloc[0]["author_id"]
        counts = grp["author_id"].value_counts()
        total = counts.sum()
        best_author, best_doa = None, -np.inf
        for author, n in counts.items():
            others = total - n
            doa = DOA_FA * (author == first_author) + DOA_LOG * np.log1p(n) + DOA_AC * np.log1p(others)
            if doa > best_doa:
                best_doa, best_author = doa, author
        if best_author is not None and best_doa > 0:
            owners[fpath] = best_author
    return owners


def truck_factor_set(file_owner: dict[str, str]) -> list[str]:
    if not file_owner:
        return []
    owned_files: dict[str, set] = defaultdict(set)
    for f, a in file_owner.items():
        owned_files[a].add(f)
    total = len(file_owner)
    remaining = set(file_owner.keys())
    tf_set: list[str] = []
    covered = 0
    while covered < 0.5 * total and owned_files:
        top_author = max(owned_files, key=lambda a: len(owned_files[a] & remaining))
        top_files = owned_files.pop(top_author) & remaining
        if not top_files:
            break
        tf_set.append(top_author)
        remaining -= top_files
        covered = total - len(remaining)
    return tf_set

## Step 3-6: TFDD detection, pre-departure diffusion, snapshot covariates, survival

`process_repo` runs the full per-repo pipeline: find the first founder-only TFDD event (TF-set of size 1, silent for `SILENCE_MONTHS`, and that one author is the repo's first committer), measure the pre-departure authority-diffusion trajectory (`founder_share_pre`, `n_diffuse_owners_pre`), record Avelino et al.'s at-TFDD snapshot covariates, classify 18-month post-TFDD survival, and draw window-shuffle placebo samples for the later placebo check.

In [ ]:
@dataclass
class RepoResult:
    repo_id: str
    language: str
    license: str
    stars: float
    forks: float
    alias_collapse_rate: float
    has_founder_tfdd: bool = False
    tfdd_date: Optional[str] = None
    founder: Optional[str] = None
    founder_share_pre: Optional[float] = None
    n_diffuse_owners_pre: Optional[float] = None
    diffusion_score: Optional[float] = None
    developers_at_tfdd: Optional[int] = None
    commits_at_tfdd: Optional[int] = None
    files_at_tfdd: Optional[int] = None
    contributor_count: Optional[int] = None
    survival_label: Optional[str] = None
    survived_binary: Optional[int] = None
    placebo_founder_shares: list = field(default_factory=list)
    placebo_n_diffuse_owners: list = field(default_factory=list)
    error: Optional[str] = None


def _year_ends(commits: pd.DataFrame) -> list[pd.Timestamp]:
    y0, y1 = commits["ts"].min().year, commits["ts"].max().year
    return [pd.Timestamp(year=y, month=12, day=31, tz="UTC") for y in range(y0, y1 + 1)]


def _first_commit_author(commits: pd.DataFrame) -> str:
    first_ts = commits["ts"].min()
    early = commits[commits["ts"] <= first_ts + timedelta(days=1)]
    return early["author_id"].value_counts().idxmax()


def classify_survival(commits: pd.DataFrame, tfdd_date: pd.Timestamp, departing_set: set) -> tuple[str, int]:
    window_end = tfdd_date + months(SURVIVAL_WINDOW_MONTHS)
    post = commits[(commits["ts"] > tfdd_date) & (commits["ts"] <= window_end)]
    pre = commits[commits["ts"] <= tfdd_date]
    if post.empty:
        return "dead", 0
    new_dev_commits = post[~post["author_id"].isin(departing_set)]
    n_new_devs = new_dev_commits["author_id"].nunique()
    if n_new_devs == 0:
        return "dormant", 0
    # regained TF set (post-window, using files touched only in the window)
    owners_post = compute_doa_owner_per_file(post, window_end)
    non_dep_owners = {a for a in owners_post.values() if a not in departing_set}
    pre_year = pre[pre["ts"] > tfdd_date - months(12)]
    pre_monthly = pre_year.groupby(pre_year["ts"].dt.to_period("M")).size()
    pre_median = float(pre_monthly.median()) if len(pre_monthly) else 0.0
    post_monthly = post.groupby(post["ts"].dt.to_period("M")).size()
    post_rate = float(post_monthly.mean()) if len(post_monthly) else 0.0
    if len(non_dep_owners) >= 2 and post_rate >= pre_median and pre_median > 0:
        return "thriving", 1
    if len(non_dep_owners) >= 1:
        return "maintained", 1
    return "dormant", 0


def process_repo(raw_repo: dict, seed: int) -> RepoResult:
    rng = random.Random(seed)
    parsed = load_repo_commits(raw_repo)
    if parsed is None:
        return RepoResult(repo_id="unknown", language="unknown", license="unknown", stars=0, forks=0, alias_collapse_rate=0, error="no_commits")
    repo_id, commits = parsed["repo_id"], parsed["commits"]
    base = RepoResult(
        repo_id=repo_id,
        language=parsed["language"],
        license=parsed["license"],
        stars=parsed["stars"],
        forks=parsed["forks"],
        alias_collapse_rate=parsed["alias_collapse_rate"],
    )
    try:
        year_ends = _year_ends(commits)
        if len(year_ends) < 2:
            base.error = "insufficient_history"
            return base
        founder = _first_commit_author(commits)

        yearly_tf: dict[pd.Timestamp, list[str]] = {}
        for ye in year_ends:
            owners = compute_doa_owner_per_file(commits, ye)
            yearly_tf[ye] = truck_factor_set(owners)

        last_commit_by_author = commits.groupby("author_id")["ts"].max()

        tfdd_year_end = None
        departing_set: list[str] = []
        sorted_years = sorted(year_ends)
        for i, ye in enumerate(sorted_years):
            tf_set = yearly_tf[ye]
            if not tf_set:
                continue
            silent = all(
                (ye - last_commit_by_author.get(a, commits["ts"].min())).days >= SILENCE_MONTHS * 30.4375
                for a in tf_set
            )
            if silent:
                tfdd_year_end = ye
                departing_set = tf_set
                break
        if tfdd_year_end is None:
            base.error = "no_tfdd"
            return base
        if len(departing_set) != 1 or departing_set[0] != founder:
            base.error = "not_founder_only_tfdd"
            return base

        tfdd_date = last_commit_by_author[founder] + months(SILENCE_MONTHS)
        min_post_needed = tfdd_date + months(SURVIVAL_WINDOW_MONTHS)
        if commits["ts"].max() < min_post_needed - months(3):
            base.error = "right_censored"
            return base

        base.has_founder_tfdd = True
        base.tfdd_date = tfdd_date.isoformat()
        base.founder = founder

        # STEP 4: pre-departure diffusion trajectory
        def diffusion_in_window(w_start: pd.Timestamp, w_end: pd.Timestamp) -> tuple[float, int]:
            wc = commits[(commits["ts"] >= w_start) & (commits["ts"] < w_end)]
            founder_share = float((wc["author_id"] == founder).sum() / max(len(wc), 1))
            doa_pre = compute_doa_owner_per_file(commits[commits["ts"] < w_end], w_end)
            owners_pre = set(doa_pre.values())
            n_diffuse = len(owners_pre - {founder})
            return founder_share, n_diffuse

        w_start = tfdd_date - months(PRE_WINDOW_FAR_MONTHS)
        w_end = tfdd_date - months(PRE_WINDOW_NEAR_MONTHS)
        founder_share, n_diffuse = diffusion_in_window(w_start, w_end)
        base.founder_share_pre = founder_share
        base.n_diffuse_owners_pre = float(n_diffuse)
        base.diffusion_score = float((1 - founder_share) * np.log1p(n_diffuse))

        # STEP 5: at-TFDD snapshot covariates
        at_tfdd = commits[commits["ts"] <= tfdd_date]
        base.developers_at_tfdd = int(at_tfdd["author_id"].nunique())
        base.commits_at_tfdd = int(at_tfdd["sha"].nunique())
        base.files_at_tfdd = int(at_tfdd["file"].nunique())
        base.contributor_count = int(commits["author_id"].nunique())

        # STEP 6: survival outcome
        label, surv_bin = classify_survival(commits, tfdd_date, set(departing_set))
        base.survival_label = label
        base.survived_binary = surv_bin

        # STEP 9: placebo draws (window-shuffle)
        earliest = commits["ts"].min()
        latest_allowed_start = tfdd_date - months(18) - months(PRE_WINDOW_NEAR_MONTHS)
        if latest_allowed_start > earliest:
            span_days = (latest_allowed_start - earliest).days
            n_draws = min(N_PLACEBO_DRAWS, 20)  # per-repo cap; aggregated across repos downstream
            for _ in range(n_draws):
                offset = rng.uniform(0, max(span_days, 1))
                p_start = earliest + timedelta(days=offset)
                p_end = p_start + months(PRE_WINDOW_FAR_MONTHS - PRE_WINDOW_NEAR_MONTHS)
                if p_end >= w_start:
                    continue
                fs, nd = diffusion_in_window(p_start, p_end)
                base.placebo_founder_shares.append(fs)
                base.placebo_n_diffuse_owners.append(nd)

        return base
    except Exception as e:  # noqa: BLE001
        base.error = f"exception: {e}"
        print(f"repo {repo_id} failed: {e}")
        return base

## Run the per-repo pipeline

Process each repo sequentially (matches the original: per-process import overhead makes multiprocessing slower than sequential for corpora this size).

In [ ]:
t_start = time.time()

repos_to_process = raw_repos[:MAX_REPOS]
results: list[RepoResult] = []
for i, rr in enumerate(repos_to_process):
    results.append(process_repo(rr, RNG_SEED + i))

n_repos_total = len(results)
founder_events = [r for r in results if r.has_founder_tfdd]
print(f"n_repos_total={n_repos_total}, n_founder_tfdd_events={len(founder_events)}")

error_counts = defaultdict(int)
for r in results:
    if r.error:
        error_counts[r.error] += 1
print(f"error breakdown: {dict(error_counts)}")

alias_rates = [r.alias_collapse_rate for r in results if r.alias_collapse_rate is not None]
alias_qa = {
    "median_collapse_rate": float(np.median(alias_rates)) if alias_rates else None,
    "n_repos_over_40pct_collapse": int(sum(1 for a in alias_rates if a > 0.4)),
}
print(f"alias_qa: {alias_qa}")
print(f"elapsed: {time.time() - t_start:.2f}s")

## Step 7-9: cross-repo analysis

Build a DataFrame of the founder-TFDD events, then:
- `matched_pairs_analysis`: nearest-neighbor matching (on standardized log-stars/log-forks/log-contributors within language) of high- vs. low-diffusion projects, bootstrap 95% CI on the survival lift.
- `run_regressions`: BH-corrected logistic + ordinal regression of survival on the diffusion predictors plus Avelino et al.'s snapshot covariates.
- `placebo_check`: refits the regression on window-shuffled (placebo) diffusion draws, to see whether the true pre-departure window's effect exceeds the null distribution of effects from arbitrary windows.
- `baseline_snapshot_predict` / `ourmethod_predict`: the two side-by-side survival predictors (Avelino et al.'s snapshot-only baseline vs. this run's diffusion-augmented method) emitted per example.

In [ ]:
def matched_pairs_analysis(df: pd.DataFrame, rng: np.random.Generator) -> dict:
    df = df.copy()
    df["log_stars"] = np.log1p(df["stars"])
    df["log_forks"] = np.log1p(df["forks"])
    df["log_contrib"] = np.log1p(df["contributor_count"])
    high = df[(df["founder_share_pre"] < 0.5) & (df["n_diffuse_owners_pre"] >= 2)]
    low = df[df["founder_share_pre"] >= 0.8]
    pairs = []
    for lang, hgrp in high.groupby("language"):
        lgrp = low[low["language"] == lang]
        if lgrp.empty:
            continue
        feats_low = lgrp[["log_stars", "log_forks", "log_contrib"]].values
        nn = NearestNeighbors(n_neighbors=1).fit(feats_low)
        feats_high = hgrp[["log_stars", "log_forks", "log_contrib"]].values
        dist, idx = nn.kneighbors(feats_high)
        for hi, (d, j) in zip(hgrp.index, zip(dist.ravel(), idx.ravel())):
            pairs.append((hi, lgrp.index[j], float(d)))
    if not pairs:
        return {"n_pairs": 0, "survival_lift": None, "ci_95": None, "p_value": None, "note": "no eligible matched pairs (relaxed matching not triggered: sample too small)"}
    lifts = []
    for hi, li, _ in pairs:
        lifts.append(df.loc[hi, "survived_binary"] - df.loc[li, "survived_binary"])
    lifts = np.array(lifts, dtype=float)
    obs_lift = float(lifts.mean())
    boot = rng.choice(lifts, size=(N_BOOTSTRAP, len(lifts)), replace=True).mean(axis=1)
    ci = (float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))
    # two-sided p-value from bootstrap null-shift (test lift != 0)
    p = float(2 * min((boot <= 0).mean(), (boot >= 0).mean()))
    p = min(p, 1.0)
    return {"n_pairs": len(pairs), "survival_lift": obs_lift, "ci_95": ci, "p_value": p}


def run_regressions(df: pd.DataFrame) -> dict:
    d = df.dropna(subset=["founder_share_pre", "n_diffuse_owners_pre", "survived_binary"]).copy()
    if len(d) < 10:
        return {"logistic": {"error": "insufficient_n", "n": len(d)}, "ordinal": {"error": "insufficient_n", "n": len(d)}}
    d["log_stars"] = np.log1p(d["stars"])
    d["log_forks"] = np.log1p(d["forks"])
    d["contributor_count_z"] = (d["contributor_count"] - d["contributor_count"].mean()) / (d["contributor_count"].std() or 1)
    lang_dummies = pd.get_dummies(d["language"], prefix="lang", drop_first=True)
    lic_dummies = pd.get_dummies(d["license"], prefix="lic", drop_first=True)
    predictors = ["founder_share_pre", "n_diffuse_owners_pre", "log_stars", "log_forks", "contributor_count_z"]
    X = pd.concat([d[predictors], lang_dummies, lic_dummies], axis=1).astype(float)
    Xz = X.copy()
    for c in predictors:
        s = Xz[c].std()
        Xz[c] = (Xz[c] - Xz[c].mean()) / s if s else 0.0
    Xc = sm.add_constant(Xz, has_constant="add")
    y = d["survived_binary"].astype(float)

    logit_out: dict = {}
    try:
        model = sm.Logit(y, Xc.astype(float))
        res = model.fit(disp=0, maxiter=200)
        pvals = res.pvalues.drop("const", errors="ignore")
        rej, p_bh, _, _ = multipletests(pvals.values, method="fdr_bh")
        logit_out = {
            "coeffs": {k: float(v) for k, v in res.params.items()},
            "se": {k: float(v) for k, v in res.bse.items()},
            "p_raw": {k: float(v) for k, v in res.pvalues.items()},
            "p_bh": dict(zip(pvals.index, [float(p) for p in p_bh])),
            "std_effect_founder_share_pre": float(res.params.get("founder_share_pre", np.nan)),
            "std_effect_n_diffuse_owners_pre": float(res.params.get("n_diffuse_owners_pre", np.nan)),
            "n": int(len(d)),
            "converged": bool(res.mle_retvals.get("converged", False)),
        }
    except Exception as e:  # noqa: BLE001
        logit_out = {"error": str(e), "n": int(len(d))}

    ordinal_out: dict = {}
    if OrderedModel is not None and d["survival_label"].nunique() >= 3:
        try:
            order = ["dead", "dormant", "maintained", "thriving"]
            cats = pd.Categorical(d["survival_label"], categories=[c for c in order if c in d["survival_label"].unique()], ordered=True)
            om = OrderedModel(cats.codes, Xz.astype(float), distr="logit")
            ores = om.fit(method="bfgs", disp=0, maxiter=200)
            ordinal_out = {
                "coeffs": {k: float(v) for k, v in ores.params.items() if k in Xz.columns},
                "p_raw": {k: float(v) for k, v in ores.pvalues.items() if k in Xz.columns},
                "n": int(len(d)),
            }
        except Exception as e:  # noqa: BLE001
            ordinal_out = {"error": str(e), "n": int(len(d))}
    else:
        ordinal_out = {"error": "insufficient_label_levels_or_no_ordered_model", "n": int(len(d))}

    # snapshot-vs-diffusion standardized effect sizes (Cohen's d equivalents via logistic beta -> d approx)
    def beta_to_d(beta):
        return float(beta * (np.sqrt(3) / np.pi)) if beta == beta else None

    snap_vs_diff = {}
    if "coeffs" in logit_out:
        for k in predictors:
            b = logit_out["coeffs"].get(k)
            snap_vs_diff[k] = {"beta": b, "cohens_d_equiv": beta_to_d(b) if b is not None else None}

    return {"logistic": logit_out, "ordinal": ordinal_out, "snapshot_vs_diffusion_effect_sizes": snap_vs_diff}


def placebo_check(df: pd.DataFrame, true_regression: dict) -> dict:
    d = df.dropna(subset=["placebo_founder_shares", "placebo_n_diffuse_owners"])
    d = d[d["placebo_founder_shares"].map(len) > 0]
    if d.empty:
        return {"error": "no_placebo_draws_available"}
    true_beta = true_regression.get("logistic", {}).get("std_effect_founder_share_pre")
    if true_beta is None or true_beta != true_beta:
        return {"error": "true_effect_unavailable"}
    n_draws = min(d["placebo_founder_shares"].map(len).min(), N_PLACEBO_DRAWS)
    placebo_effects = []
    rng = np.random.default_rng(RNG_SEED)
    for draw_i in range(int(n_draws)):
        pdf = d.copy()
        pdf["founder_share_pre"] = pdf["placebo_founder_shares"].map(lambda lst, i=draw_i: lst[i] if i < len(lst) else np.nan)
        pdf["n_diffuse_owners_pre"] = pdf["placebo_n_diffuse_owners"].map(lambda lst, i=draw_i: lst[i] if i < len(lst) else np.nan)
        preg = run_regressions(pdf)
        b = preg.get("logistic", {}).get("std_effect_founder_share_pre")
        if b is not None and b == b:
            placebo_effects.append(float(b))
    if not placebo_effects:
        return {"error": "placebo_regressions_all_failed"}
    placebo_effects = np.array(placebo_effects)
    frac_ge = float((np.abs(placebo_effects) >= abs(true_beta)).mean())
    return {
        "true_effect": float(true_beta),
        "placebo_null_distribution_summary": {
            "mean": float(placebo_effects.mean()),
            "std": float(placebo_effects.std()),
            "p5": float(np.percentile(placebo_effects, 5)),
            "p95": float(np.percentile(placebo_effects, 95)),
            "n_draws": int(len(placebo_effects)),
        },
        "fraction_placebo_ge_true": frac_ge,
    }


def baseline_snapshot_predict(d: pd.DataFrame) -> pd.Series:
    """Baseline = logistic regression on snapshot covariates only (developers,
    commits, files at TFDD + size), no pre-departure diffusion trajectory."""
    dd = d.dropna(subset=["survived_binary"]).copy()
    if len(dd) < 10:
        return pd.Series(index=d.index, dtype=float)
    dd["log_stars"] = np.log1p(dd["stars"])
    dd["log_forks"] = np.log1p(dd["forks"])
    X = dd[["developers_at_tfdd", "commits_at_tfdd", "files_at_tfdd", "log_stars", "log_forks"]].astype(float)
    Xc = sm.add_constant(X, has_constant="add")
    y = dd["survived_binary"].astype(float)
    try:
        res = sm.Logit(y, Xc).fit(disp=0, maxiter=200)
        pred = res.predict(Xc)
        return pred.reindex(d.index)
    except Exception:  # noqa: BLE001
        return pd.Series(index=d.index, dtype=float)


def ourmethod_predict(d: pd.DataFrame) -> pd.Series:
    dd = d.dropna(subset=["survived_binary", "founder_share_pre", "n_diffuse_owners_pre"]).copy()
    if len(dd) < 10:
        return pd.Series(index=d.index, dtype=float)
    dd["log_stars"] = np.log1p(dd["stars"])
    dd["log_forks"] = np.log1p(dd["forks"])
    X = dd[["founder_share_pre", "n_diffuse_owners_pre", "developers_at_tfdd", "commits_at_tfdd", "files_at_tfdd", "log_stars", "log_forks"]].astype(float)
    Xc = sm.add_constant(X, has_constant="add")
    y = dd["survived_binary"].astype(float)
    try:
        res = sm.Logit(y, Xc).fit(disp=0, maxiter=200)
        pred = res.predict(Xc)
        return pred.reindex(d.index)
    except Exception:  # noqa: BLE001
        return pd.Series(index=d.index, dtype=float)

In [ ]:
df = pd.DataFrame([r.__dict__ for r in founder_events]) if founder_events else pd.DataFrame(
    columns=["repo_id", "language", "license", "stars", "forks", "founder_share_pre", "n_diffuse_owners_pre",
             "developers_at_tfdd", "commits_at_tfdd", "files_at_tfdd", "contributor_count", "survived_binary", "survival_label"])

rng = np.random.default_rng(RNG_SEED)
matched_pairs = matched_pairs_analysis(df, rng) if not df.empty else {"n_pairs": 0, "error": "no_founder_tfdd_events"}
regression = run_regressions(df) if not df.empty else {"logistic": {"error": "no_founder_tfdd_events"}, "ordinal": {"error": "no_founder_tfdd_events"}}
placebo = placebo_check(df, regression) if not df.empty else {"error": "no_founder_tfdd_events"}

if not df.empty:
    df["predict_baseline_prob"] = baseline_snapshot_predict(df)
    df["predict_ourmethod_prob"] = ourmethod_predict(df)

print("matched_pairs:", json.dumps(matched_pairs, indent=2, default=str))
print("\nregression.logistic:", json.dumps(regression.get("logistic", {}), indent=2, default=str))
print("\nplacebo_check:", json.dumps(placebo, indent=2, default=str))

## Results

Table of the founder-TFDD events found and a scatter plot of pre-departure founder commit-share vs. number of distinct non-founder DOA owners, colored by whether the project survived (thriving/maintained) or not (dormant/dead) the 18-month post-departure window. Diffuse-authority repos (low founder-share, several owners) should cluster toward the surviving side.

In [ ]:
display_cols = ["repo_id", "language", "founder_share_pre", "n_diffuse_owners_pre", "diffusion_score",
                "developers_at_tfdd", "commits_at_tfdd", "files_at_tfdd", "survival_label", "survived_binary"]
if not df.empty:
    from IPython.display import display
    display(df[display_cols].round(3))
else:
    print("No founder-only TFDD events found in this demo sample.")

fig, ax = plt.subplots(figsize=(7, 5))
if not df.empty:
    colors = df["survived_binary"].map({1: "tab:green", 0: "tab:red"})
    ax.scatter(df["founder_share_pre"], df["n_diffuse_owners_pre"], c=colors, s=120, edgecolor="black", zorder=3)
    for _, row in df.iterrows():
        ax.annotate(row["repo_id"].split("/")[-1], (row["founder_share_pre"], row["n_diffuse_owners_pre"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=8)
    from matplotlib.lines import Line2D
    legend_elems = [Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:green", markeredgecolor="black", markersize=10, label="survived (thriving/maintained)"),
                    Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:red", markeredgecolor="black", markersize=10, label="did not survive (dormant/dead)")]
    ax.legend(handles=legend_elems, loc="upper right")
ax.set_xlabel("founder commit-share, 6-12mo pre-departure")
ax.set_ylabel("distinct non-founder DOA owners, 6-12mo pre-departure")
ax.set_title("Pre-departure authority diffusion vs. 18mo post-TFDD survival")
ax.grid(alpha=0.3, zorder=0)
plt.tight_layout()
plt.show()

print(f"\nn_repos_total={n_repos_total}  n_founder_tfdd_events={len(founder_events)}")
print(f"matched_pairs n_pairs={matched_pairs.get('n_pairs')}  survival_lift={matched_pairs.get('survival_lift')}")
print(f"placebo_check: {placebo}")